In [4]:
    from qdrant_client import QdrantClient,models
    import numpy as np
    from faker import Faker

    fake = Faker()
    points = []
    for i in range(1,101):
        vector = np.random.rand(128).tolist()
        payload = {
            "name": fake.name(),
            "address" : fake.address(),
            "email" : fake.email()
        }
        point = models.PointStruct(id = i,vector=vector,payload=payload)
        points.append(point)
    client = QdrantClient("localhost",port=6333)
    collection_name = "my_collection"

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=128,distance=models.Distance.COSINE)
        )
        print(f"Collection '{collection_name}' created successfully")
    else:
        print(f"Collection '{collection_name}' already exists.")

    client.upsert(collection_name=collection_name,points = points,wait=True)
    print(f"Successfully inserted {len(points)} records into '{collection_name}'.")

    search_vector = np.random.rand(128).tolist()
    results = client.query_points(
        collection_name=collection_name,
        query=search_vector,
        limit=2,
        with_payload = True,
        with_vectors = False
    )
    print("\nQuery Results:")
    for result in results.points:
        print(f"ID: {result.id}, Score: {result.score:.4f}, Payload: {result.payload}\n")

Collection 'my_collection' already exists.
Successfully inserted 100 records into 'my_collection'.

Query Results:
ID: 20, Score: 0.8400, Payload: {'name': 'Geoffrey Hill', 'address': '8027 Kristine Shoals\nLoganbury, FM 66730', 'email': 'joshuabrooks@example.com'}

ID: 39, Score: 0.8144, Payload: {'name': 'Timothy Adams', 'address': '3586 Alison Station\nJessicaberg, CO 79198', 'email': 'stephenhicks@example.net'}

